# Abstract

The `llm-workbench` Python library is a toolkit designed to guide data scientists and machine learning practitioners through the process of evaluating multiple LLMs for performance on custom tasks, including agentic tasks. While some libraries already exist to assist with certain parts of this kind of workflow, this library combines the necessary stages into a single utility, reducing the overhead for users, and also reducing the risk of omitting a key step required to achieve a rigorous result. 

Evaluating LLMs along multiple metrics, instead of choosing a single evaluation alone, is vital because performance of LLMs is highly variable along multiple fronts. In addition, conducting evaluations with statistical power is key because the non-deterministic nature of LLM behavior can lead to wide swings in performance from case to case. This library guides users in creating evals that cover all the attributes of LLM performance that are relevant to their use case, calibrating LLM-as-a-judge performance for accuracy, and creating an experiment large enough to detect an effect reliably, including providing easy-to-use analysis reports illustrating that significance and the pertinent outcomes. 

# Introduction

Businesses and organizations are increasingly finding mission critical use cases for large language models. However, at the same time, the cost of tokens is becoming burdensome on budgets, exacerbated by the uncertainty inherent in the technology - a user cannot predict how many tokens any LLM will use to answer a question, nor whether repeated attempts will be required to get the desired result. Because of this, and because of their noticeable performance improvements, open weight and open source LLMs are gaining in popularity. Essentially, users want to shop around for the best deal when choosing the LLM they will use.

However, price is not the only consideration that matters. To make informed decisions about whether switching models is acceptable, users need to conduct statistically rigorous tests of performance (@sec-methodology), and that is what `llm-workbench` enables.

Testing LLM performance and comparing it effectively is quite difficult. An LLM will give different responses to the same prompt when repeatedly run, which is inherent to the non-deterministic nature of the technology. For many tasks, there is no conclusive ground truth to test against, and if there is, it is not the only consideration that is important for judging performance quality. `llm-workbench` provides methods that help isolate performance in light of these realities. It sets up the tests as paired-prompt measurements, and also includes tools for ensuring that an effective experiment can be produced, avoiding spurious conclusions.

There are some other Python libraries (@sec-other-libraries) available that address parts of the problem - some that help with evals, some that offer access to different statistical tests, but at this time none are comprehensive guides through the end-to-end process in the way that `llm-workbench` is. The purpose of this library is specifically to support testing multiple LLMs on the same task and rigorously determine which one/s perform best, to enable informed choices. 


# Background & Motivation

## Contrasting with "Vibes"

Teams working with LLMs today often want to have a method of testing their performance. For example, when a new version of their existing LLM of choice is released, it makes sense to check that it will be as good as or better than the version they're accustomed to. However, many teams struggle to know how to confidently run such tests, and may fall back to spot checks or a handful of gold standard prompts where the responses are eyeballed by human observers. These methods leave us open to failing to recognize real performance regressions or shifts, because they are not comprehensive or statistically rigorous.

The same issue comes up in the scenario described in the introduction. Budgets are straining under the weight of tokens, but without a method of studying alternative, cheaper LLMs that feels safe, teams often cling to the high-cost LLMs from frontier labs, because the risk of switching seems untenable. 

There are, of course, GUI frameworks that more mature teams may use to maintain and monitor evaluations (@sec-related), and this library does not intend to compete with these for the most part. Routine day to day monitoring of model behavior is different from the experimental testing to choose which model you want to employ. Some of these SaaS offerings do have elements that can compare models with one another, but they often have a rigidity to their structures that make bringing custom (especially agentic) tasks and evals cumbersome. `llm-workbench` is a lightweight tool with components that are more flexible for customized needs.


## Maintaining Components as Code

The development framework used here assumes all components should be maintained as code to the highest degree possible. This makes versioning and maintenance of the evaluations, prompts, and other components easier, and enables reproducibility and testing. In addition, the components are portable and shareable, so that if users want to copy the evaluations to or from a production-level monitoring tool, the effort required to make the transition should be minimal. Evals are `eval.yaml` + `prompt.md` files, datasets are jsonl, and tasks are Python modules, all of which the user maintains and can manipulate at will.

## Design Goals

This library is meant to allow users to follow a streamlined workflow for the LLM testing process. However, it takes no opinionated stance on what kind of evaluation criteria the user employs. There are scaffolding templates for evaluations ("contains" and LLM-as-a-judge) as well as extension points where the user may build out significant complexity if desired (@sec-extensibility). Similarly, the library has no opinion about what kind of tasks the user might ask the LLM to do. This is also an area where simple forms are provided, but extensibility is inherent to the project. The author has used this library for testing complex agentic workflows, by creating custom tasks that call tools and conduct multi-step workflows, but this library does not provide detailed instruction on this piece. The important thing is that if you are testing an LLM as part of a complex agentic workflow, that can be dropped in to this framework and the components should work just as if a simple one-step LLM task was involved.


# Core Concepts {#sec-concepts}

It may help users getting started to review the key concepts, some of which have already been mentioned in passing.

## Evaluations

Evaluations, or "evals", in this context are standards of performance for the LLM. Fundamentally, the user must think about what quality performance looks like, and how they would determine it, in order to create their evals. These can be a combination of many different measures, from factual tests (does the response contain value X) to tonal or stylistic judgments (is the response warm and friendly). Different types of evaluation require different types of judge, as will be explained below. The evaluation is the core, however, and much time and thought should be given to creating these so that the rest of the experiment is aligned with the project's goals.

## Judges

Judges in this case are different ways to measure the evaluation - "LLM as a judge", or testing against ground truth of some kind. If the evaluation is a factual test, where two strings can be simply compared (does the response contain value X), then a deterministic judge is ideal. See @sec-casestudy for details about how to implement each.

* `contains` — passes if the output includes a specific substring or field value. Good for checking that required keywords, labels, or identifiers appear.
* `exact_match` — passes only if the output matches the expected value exactly. Most useful for tightly constrained outputs like codes, single-word answers, or enum values where there is genuinely only one correct form.
* `regex` — tests the output against a regular expression. Useful when the correct answer has a known structure but some variability in form — a date format, a phone number, a currency string.
* `json_schema` — validates that the output is valid JSON and conforms to a supplied schema. The right tool when you're asking the LLM to produce structured output and need to verify the shape and types of the result.
* `numeric_tolerance` — checks whether a numeric value in the output falls within a specified range of the expected value. Useful for calculations or extractions where small rounding differences are acceptable.

Alternatively, a subjective assessment, which is often required for LLMs because of their non-deterministic nature, will use another LLM to judge. Calibrating the LLM judge is a distinct part of the process that is also important, because the LLM judge is standing in as proxy for the human user. The `llm-workbench` library offers tooling to assist with this calibration as well (@sec-calibration). Users can also generate custom judges if they need more specific customized versions (@sec-extensibility).

## Datasets

Datasets are the individual cases that you will use for testing the LLM. These may be totally distinct prompts, such as factual questions, or they may be details that are populated into a larger prompt template.

## Tasks

Tasks are the work you actually want your LLM to perform, using the dataset records. The built-in `PromptTemplateTask` provides a starting point where input values from the dataset fill a prompt template. If the task needs to be more complex than one call to an LLM, custom tasks can do anything, including multi-step agentic work, and always return a `TaskOutput`.

## Models

Models are LLMs being compared, named as backend/name (e.g. ollama/llama3.1). These use the same mechanics in the backend as the judge LLM, which scores outputs rather than being tested.

## Runs

A run is a single execution of the model x prompt case sweep. Every model under test answers every prompt in the dataset, producing the results the analysis works from. 

## Metrics

A metric is the score a judge produces for one model on one eval. We'll use these in this framework as inputs to the analytics (@sec-methodology), to determine how well each LLM performs in aggregate. 


# Related Tools {#sec-related}

There are quite a few tools in the Python ecosystem that address parts of the LLM comparison and evaluation process. Often, these focus on prompt engineering and scrutinizing the performance of different models on specific prompts, taking a very granular view of the work. `llm-workbench` uses a different philosophy, adapting the statistical methods used in various scientific fields to produce recommendations across a composite of quality. They range in other ways, some being very heavyweight platforms with lots of setup and maintenance required (Braintrust, LangSmith, Arize Phoenix) while others have less internal complexity or bulk to manage. Many of the tools that are closest in overlap with functionalities found in `llm-workbench` tend to be the heaviest, so if users want to run experiments but avoid a lot of overhead and setup, this is important to note. In several cases, this is just because the platforms cover a huge scope of functionalities that are beyond this specific project. 

In addition, some of these frameworks come with more wrappings around the LLM tasks themselves. If starting from zero, this can be handy, but if a user is trying to produce an experiment that replicates their production LLM/agentic task, this can be a hassle. `llm-workbench` strongly avoids exercising any control over the tasks brought to the experiment, so that users can control this themselves.


## Other Frameworks {#sec-other-libraries}

**DeepEval**

* https://deepeval.com

DeepEval is a very popular and well regarded open source toolkit for designing and running LLM evals in Python, offering a large library of ready-made LLM-as-a-judge metrics (spanning RAG, agentic, and safety checks) and flexible integration into existing LLM and agentic workflows. Its scope is simply designing and running evals, but the analysis, calibration, and multi-model comparison are not part of DeepEval. As with some other frameworks discussed here, DeepEval could be extremely helpful for users who need more assistance with eval design and engineering. `llm-workbench` could complement DeepEval by serving as a statistical analysis layer.


**Inspect**

* https://inspect.aisi.org.uk 

Inspect is an LLM framework developed by the UK AI Security Institute, which offers quite a bit of overlap with `llm-workbench`. Both libraries use a similar object architecture, in this case Task/Dataset/Scorer (`llm-workbench` has Task/Dataset/Judge). Inspect reports descriptive measures of uncertainty (standard errors) but lacks the statistical tooling llm-workbench centers on (power analysis, inferential or comparative testing), and doesn't offer LLM-as-a-judge calibration. In addition, `llm-workbench` provides mixed-effects comparisons, which let you compare multiple models in concert, instead of one by one or pairwise only. For users looking for a low lift start, `llm-workbench` provides an easier path to entry, because the CLI enables a basically no-code starting point, while Inspect is Python-first, but in cases with increased complexity, such as custom tasks, both require similar effort from the user. 



**Arize Phoenix**

* https://arize.com/phoenix/ 

While Arize is a commercial product platform, they offer an open source evaluation toolkit called Phoenix. It's built on top of OpenTelemetry and OpenInference, and can be self hosted. It's more comprehensive for agent management, monitoring, and debugging, areas that are outside the scope of `llm-workbench`, but does not cover the inferential statistical analysis and experiment design space. Unlike the other offerings discussed, it does have tooling for LLM-as-a-judge calibration, so users who need a heavier full platform that includes this may find Phoenix helpful.  


**LLM Comparator**

* https://github.com/PAIR-code/llm-comparator
* https://ieeexplore.ieee.org/abstract/document/10670495 
* retired in Feb 2026

The LLM Comparator project, recently sunsetted by the Google PAIR team, is an example of a tool for comparing and contrasting LLMs on tasks that uses a granular view to tell users why model responses vary. It provides some very useful tooling, such as interactive visualization, that a lightweight library like `llm-workbench` doesn't. It allows users to do very precise scrutinizing of LLM responses to specific prompts, but is less robust when it comes to aggregate analysis, and contains minimal statistical evaluation tooling (nothing about significance testing, effect sizes, or power analysis, among other things). It also limits comparisons to 2 models at a time, which makes users do a significant amount of extra analytical work when looking at a larger pool of models. Notably, it doesn't include any tooling for LLM-as-a-judge calibration against human reviewers (@sec-calibration), which is extremely important for LLM eval work. For exploratory work limited to comparing two models that seeks to deep dive into specific prompts, LLM Comparator would be very useful, and that is a different functional role than `llm-workbench`. 


**promptfoo**

* https://github.com/promptfoo/promptfoo
* https://www.promptfoo.dev/docs/
* in process of being acquired by OpenAI in 2026

This project covers a much broader scope than `llm-workbench`, but has potential complementarity for certain workflows. It is an eval and red-teaming toolkit weighted toward security, including side-by-side model comparison across many providers, a web viewer, CI/CD integration, and extensive vulnerability-scanning tooling. Because `llm-workbench` is narrowly focused on statistical experimentation for comparing LLMs, there is a relatively small space of overlap around designing and running LLM-as-a-judge and deterministic evals. `promptfoo` does not engage with inferential statistics or mixed effects modeling, and offers no tooling for LLM-as-a-judge calibration, but it could serve as the task harness producing runs for evaluation, with `llm-workbench` providing both the calibration and statistical analysis components that the other library lacks. 


**ChainForge**

* https://dl.acm.org/doi/full/10.1145/3613904.3642016
* https://github.com/ianarawjo/ChainForge
* Released 2023

ChainForge is a visual, node-based GUI toolkit for exploring model behavior, letting users interactively investigate and compare LLM responses across prompts, models, and settings. However, it doesn't include LLM-as-a-judge calibration, nor does it contain the statistical tests that `llm-workbench` provides, meaning that users will not get the information needed to judge the significance of differences from ChainForge. ChainForge centers on a visual node/flow representation, while `llm-workbench` keeps evals, prompts, and datasets as version-controllable code. Users who need to conduct prompt engineering and exploration may find ChainForge helpful, but it doesn't overlap with `llm-workbench` in statistical functionality.


## Commercial Tools

Commercial SaaS platforms occupy an adjacent but distinct space: they emphasize production observability, tracing, human-feedback collection, and eval dashboards across the application lifecycle, typically as hosted, proprietary services. They generally prioritize monitoring and iteration over inferential statistics or judge calibration, and because they are closed source and hosted, they differ in kind from the open, code-first tooling discussed above and from `llm-workbench`'s focus on statistical model selection.

**Braintrust**

* https://www.braintrust.dev

Braintrust is a commercial product for LLM and agent observability/evaluation, meant to be used for monitoring production traffic primarily, but it also includes playground and side-by-side experiment structures. As with other tools discussed here, it lacks deeper statistical reporting and LLM-as-a-judge calibration. It doesn't overlap much with `llm-workbench`'s main purpose but instead is targeted to users looking to purchase a tool for production LLM observability. 


**LangSmith**

* https://www.langchain.com/langsmith/evaluation 

LangSmith is a platform in the LangChain ecosystem. It offers observability and evaluation, tracing, LLM-as-a-judge evaluators, and pairwise experiment-based comparison of prompts and models. Like Braintrust, it is a proprietary, hosted service oriented toward monitoring, and it reports aggregate evaluator scores rather than inferential statistics, with no LLM-as-a-judge calibration. It overlaps with `llm-workbench` on running judged evals and comparing models, but not on the statistical rigor or calibration that `llm-workbench` specializes in.




<!-- 
dismissed
https://arxiv.org/html/2406.07545v1
https://arxiv.org/html/2408.13006v2
 -->


# Architecture & Design {#sec-architecture}

This library runs every candidate model over every prompt in a dataset, scores the
outputs with configurable evals, and applies a paired-statistics battery to
declare a winner. Everything is driven through one CLI (`llm-workbench`).

## System Diagram

![](architecture-overview.png)

## Five Layers

1. **CLI** (`cli/`) - seven command groups over a Click app; Rich output helpers.
2. **Orchestration** (`runner/`) - `run_comparison()` sweeps model x prompt case,
   fail-fast on bad model ids, crash-safe/resumable writes to `results.jsonl`.
3. **Three pillars run per row:**
   - **Tasks** (`tasks/`) - produce a `TaskOutput` (the model-under-test call),
     or `ReplayTask` scores a pre-existing answer.
   - **Models** (`models/`) - a prefix registry fans `make_client()` out to six
     backends; SDKs are lazy-imported so the base install stays lean.
   - **Evals + Scoring** (`evals/`, `scoring/`) - an `EvalSpec` drives an LLM
     judge, deterministic programmatic checks, and ground-truth overrides.
4. **Results store** - `results.jsonl` → `results.csv`, deduped to latest per
   (model, record).
5. **Post-Run Consumers**
    - **Review + Calibration** (`reviewing.py` + `calibrate` command) - replays a 
    run's judge scores for human review, persists reviewer scores to `review.jsonl`, 
    and reports judge-vs-human bias and agreement. The I/O-free helpers live in 
    `reviewing.py`; the interactive loop is in the CLI.
    - **Analysis** (`analysis/`, behind the `analysis` extra) - long-format frames →
   a paired/repeated-measures stats battery → `analysis.json/.md`, plots, power.

A shared spine (`object_types.py`, `config.py`) carries provider-agnostic value
types and env config through every layer. Downstream app repos
(`ai-checker-evals`, `context-graph-evals`) depend on `llm-workbench` as a git dependency
and reuse only a thin public surface (`fill_template`, `GenerationRequest`,
`TaskOutput`, `Task`), shipping their own custom `Task`, evals, and datasets.


## Key Components

**CLI**

The CLI is the primary user surface. There are seven command categories included in the CLI (see diagram) which enable interacting with the different stages of the workflow.

**Datasets**

Users will either use the CLI or manually create a case dataset to use the framework. This should be jsonl data, and each line's json object includes all the input data that the LLM would receive that is specific to a single case in a live call in the production setting. This does not include the overall prompt instructions ("You are an analyst, your output format should be XYZ") but should just be the specific details that differentiate each case from one another.

**Evals and Specs**

Evals are declared definitions of the aspects of assessment you wish to cover in your experiment. Examples may include cases like "response does not contain [banned keyword or phrase]", or comparisons between the LLM's response text and some other input data element. 

Each eval is defined with its own folder, containing two files: `eval.yaml` and `prompt.md`. The format of these evals as code is very similar to how evals are formatted for other toolkits, such as Arize, so evals previously defined elsewhere in code can be relocated to this framework with minimal difficulty.

**Tasks**

Tasks are the core component of what you are asking the LLM to do. There is a built in class called `PromptTemplateTask` that you can start from, or custom tasks can be easily created as well. All tasks must return a `TaskOutput` type object, which is defined in the framework.

**Models**

This toolkit supports six different model backends (including a "mock" for easy testing), such as AWS Bedrock (regular and Mantle), Ollama, OpenAI, and Anthropic. If there's demand, the number of backends supported may be extended in future, but for now users can extend with classes for their own backends, as shown in @sec-custom-models. The hookups require tokens and authentication details to be stored in the `.env` file by the user, and then models are called by specifying their backend and name in the CLI call, such as `bedrock-mantle/google.gemma-4-31b`. Errors fail softly so that if an LLM call has a one-off problem, the entire run is not killed.

**Scoring**

The scoring component takes the evals that have been created and runs them over the LLM responses. This happens immediately after LLM calls, so the process flow creates judge or evaluation results immediately upon receiving the task LLM response. 

Several kinds of evaluation scoring are supported, including LLM-as-a-judge (using the same model access framework as described above), deterministic or programmatic heuristics (such as looking for a key word or the absence of one), or ground-truth comparisons (where the ground truth is provided in the prompt dataset). Latency and token counts (in and out) are also recorded and saved as part of the scoring process, which can be very useful for comparing models in terms of operational considerations.

**Runner**

This is the orchestration component, which organizes and processes the model x prompt case sweeps. It builds the task model client/s (the user can select as many as they want for the tests) and the judge model client initially, then runs each task and the eval scorers. Creating the model clients up front enables fail-fast, so that if any model is unreachable no work is done before discovering this. The output is written to jsonl and is resumable in the case of early stoppage. In addition, the user may add a `--limit` flag which will run only the selected number of cases, which is helpful for smoke testing.

**Results Store**

The runner appends to a results.jsonl file as it runs, and upon conclusion processes this file to a results.csv for a flattened version that can be easily used for analysis. The result is deduped in case a model x prompt case has been run multiple times, taking the most recent model x prompt case pair. 

**Calibration**

An LLM judge is only trustworthy if its scores align with what a human reviewer would say, so the library treats calibration as a first-class step rather than an afterthought. This component replays a completed run's judge decisions for human review: for each decision it surfaces the input, the model's answer, and the judge's score and reasoning, and the reviewer inputs their own score on the same rubric. From the collected scores it computes calibration statistics, such as judge-versus-human bias and agreement rates, which can reveal whether the LLM judge is aligned with the reviewer's intent. 

**Analysis**

This component contains the code to run various statistical tests and mixed effects model, as well as creating the reports, the plots, and the power analysis. The statistical tests are described in detail in @sec-methodology. If the user is not interested in complex analysis, the library can be installed without the optional `analysis` dependencies.

**Shared Spine**

The `GenerationRequest, Response, Usage` object classes contained here provide a decoupled vocabulary, ensuring smooth interaction between the components. There is also a `config.py` file which loads credentials and provider settings from `.env` for whatever model backend is in use.


## Customization and Extensibility Points {#sec-extensibility}

One of the core design goals of llm-workbench is that it adapts to your task rather than forcing your task into its shape. Customization ranges from simply supplying your own data to extending the library's internals, and the effort required scales accordingly.


### Configuration

At the lowest lift, you change what is tested without altering the framework.

* Bring your own prompt cases: the dataset is a jsonl file where each line carries the case-specific inputs that fill your prompt template. Writing a dataset that reflects your real production inputs is the single most impactful thing you can do, and it requires no code.
* Choose the models being tested: any supported backend and model can be named on the command line (backend/name), and you can compare as many at once as you like. Swapping or adding a candidate model is a flag change.
* Choose the judge model: the LLM-as-a-judge model is selected independently of the models under test, so you can pair a strong judge with cheaper candidates. This too is just a flag.
* Bring or construct evals: each eval is a small folder - an `eval.yaml` and, for judged evals, a `prompt.md`. Because the format is declarative, you can assemble a full evaluation suite, mixing deterministic checks, ground-truth comparisons, and LLM-judged rubrics, without writing any Python. Evals authored for other tools can often be ported in with minimal change. There is also a wizard in the CLI that will guide you through simple eval design.


### Code extensions
The more complex customizations involve writing Python against the library's interfaces. This is where agentic and unusual workflows are accommodated.

* Write a custom task: when the work is more than a single templated call - multi-step reasoning, tool use, parsing structured fields out of the output - you supply a task class that does whatever you need and returns a TaskOutput. This is the primary extension point for agentic evaluation, and it slots into the same runner and scoring machinery as the built-in task.
* Write a custom scorer: beyond the built-in ground-truth and LLM-judge scorers, you can implement your own scoring logic for cases the defaults don't express - for instance, matching several values at once or checking a value in context.
* Add a new model backend: at the deepest level, you can extend the set of supported providers by implementing a client against the library's model interface and registering it in the backend registry. This is the only customization that changes the library itself rather than your project, and it benefits every task and eval once added.


## Reuse and Interaction

The structure of the library enables loose coupling and the extensibility described in @sec-extensibility. For example, a task builds the `GenerationRequest` class object and returns a `TaskOutput`, but needs no visibility into the associated backend, such as which LLM was called. Judges read the `TaskOutput` but have no need to touch any of the upstream components. These interfaces let the different stages of the workflow maintain interoperability but avoid unnecessary dependencies.

Similarly, the use of the `results.jsonl` file allows tasks and analysis to communicate without creating dependency. The task runner writes to the file, and analysis reads from the file, and they can maintain separate spheres of influence otherwise. 

This library also only enforces requirements about the structures that matter for completing the tasks. Projects that use `llm-workbench` can just import the library (including optional modules as needed), and plug their customized pieces into the broader CLI. Each user will create their own domain/case specific tasks, evaluations, and datasets, and provided the format matches the scaffold, these should work seamlessly with the provided toolkit. 


## Design Decisions/Trade-offs

There are many ways that a framework for this kind of project could be structured, and in this case, some decisions were made to preserve the key considerations - flexibility and methodological rigor.

To ensure that the project can be flexible enough to apply to the wide variety of LLM task types that we see in practice (and unknown future ones), the task design and construction is only thinly scaffolded, and anything more complex is left to the user. This may require the user to have more familiarity with developing agentic LLM workflows but it makes the overall tooling much more versatile.

Similarly, evals are left primarily to the user to design, with a basic default premise but not much detail. Choosing the right evals for a project is a vitally important part of LLM experiments, but we can't presume to know what measures of performance for any given task will be the right fit for the user. Everyone has to determine what defines quality and success for their own project.

We do, on the other hand, offer relatively strong opinions about the statistical methods (@sec-methodology). Users intending to run one generic eval and call it a day will not get the most out of this framework. Methods in this library work best when multiple complementary characteristics of the larger concept of quality are defined and assessed, giving a robust, multidimensional view of how the LLM behaves. Paired design is provided because we want to reduce the degree to which variability among prompts skews or confounds the statistical measure of LLM performance. If the user does not run the same set of prompts with every LLM, once again, this library's benefits will not be fully realized. 

A few forms of non-parametric tests are also provided, based on experience. There is a tendency, as LLM performance generally improves, for the best models to have ceiling-skewed results on certain evals. This doesn't make those evals less useful, but it does mean that scoring is not normally distributed, and tests should account for that. 

One other design decision was to make LLM-as-a-judge an encouraged scoring choice in this library. LLM evaluation is a deeply challenging venture, and traditional tools focused on measuring only against a singular ground truth miss large swaths of the variation in LLM performance. Using LLMs as judges is widely accepted in AI research, but it requires its own due diligence to maintain rigor. For that reason, `llm-workbench` includes the calibration module (@sec-calibration), where users can run checks to ensure the LLM judge behaves according to their preferences and interprets the judge prompts correctly. LLM-as-a-judge without calibration is inadequate for making assessments of task LLM behavior, and calibration is a first-class element of this library in recognition of that. 


# Statistical Methodology {#sec-methodology}

The premise of this library is that LLM performance can be effectively tested with a paired prompt comparison strategy. This means that every LLM is fed the same pool of cases, and the differences in their behavior across the spectrum of the designated evals are what we intend to measure. As a result, we are able to eliminate variance due to differences in prompt difficulty, and can attribute the differences to the actual LLM performance.

How the evaluations are designed is very important to achieving a rigorous measure of quality in this way. Evals should have thoughtfully designed spectra of performance, and should not have significant overlap in the scope of what they measure. Our goal is to produce measurement validity over the broad area of "LLM quality" on the specific task in question. This will have many attributes in most cases, but combining them together to create the composite of "quality" with high construct validity is our objective.

## Designing the Construct through Evaluations
Each evaluation you write produces scores on its own scale - a graded rubric might run 1–5, while a ground-truth check is binary. To combine them into a single measure of quality without letting the widest-ranged metric dominate, we standardize each metric independently: every score is re-expressed as a z-score, i.e. how many standard deviations it sits above or below that metric's own mean across the run. 

On this common scale, a single standard deviation shift means the same thing whether it came from the rubric or the accuracy check. The composite quality score for a given model on a given prompt is then simply the mean of its standardized metric scores. Metrics that show no variation across the run - every model scoring identically - are dropped rather than counted as zeros: a metric that cannot distinguish any of the models carries no signal about quality, and including it would only dilute the composite. 

At this time, the framework weights all metrics equally in constructing the composite, but adjusting weights may be supported in the future. If a user wants to construct the composite excluding a metric, to test the effect on the overall analysis, `--exclude-metric` is an available parameter on the analysis call. 

## Statistical Tests Included

### Power Analysis

A rigorous comparison depends on having enough cases to detect a real difference between models when one exists. Too few prompts and a genuine performance gap can hide inside the noise; the test returns "no significant difference" not because the models are alike, but because the experiment was underpowered. The power command helps you determine the size of the study that will be required to get conclusive results. 

### Do the models differ?

When there are three or more models with results generated, we begin by examining whether there is a difference detectable. For each prompt we rank the models best-to-worst, and the Friedman test asks whether one model tends to rank consistently higher than the others across the whole prompt set. It is the paired, rank-based counterpart to a one-way ANOVA, and because it operates on ranks it makes no assumption that the scores are normally distributed. 

Additionally the Kendall's W is calculated, giving an effect size from 0 (models ranked essentially at random relative to one another) to 1 (every prompt produces the identical ranking), which tells us how consistent the ordering is, not just whether it is detectable. 

### What are the differences?

The next step does unordered paired difference analysis between each pair of models. The Wilcoxon signed-rank test is the paired, non-parametric analog of a paired t-test, which looks at whether the per-prompt differences between two models lean consistently in one direction. 

With three or more models this serves as the post-hoc following Friedman; with exactly two models it is the primary test. Because comparing many pairs inflates the chance of a false positive, we apply the Holm correction to the p-values, which controls the family-wise error rate while remaining more powerful than the more familiar Bonferroni adjustment. For each pair we report the higher-scoring model, the corrected p-value, and whether the difference clears significance.

### The Size of Difference

Up to now we have calculated whether the models differ, and looked at them in a round-robin pairing fashion. However, a mixed-effects model is the next step to tell us the size and direction of the difference. This model fits score ~ model, with a random intercept for each prompt. The random intercept is what encodes the pairing: it lets every prompt sit at its own baseline difficulty and asks whether a given model scores consistently above or below that prompt's baseline. 

The output is, for each model, its estimated difference from the reference model, with a confidence interval and a p-value. When more than one metric feeds the composite, the model also includes a term for the metric so that differing metric baselines do not distort the estimates. This is the one test we compute in every run, even in the compact output mode, because the per-model differences-from-reference are the headline result most users are after.

### Standardized Effect Size

For each pair of models we also report Cohen's d, which is the mean of the paired differences divided by the standard deviation of those differences. It is important that the denominator is the standard deviation of the differences, not of the raw scores - this is the correct form for a paired design, and it expresses the typical gap between two models relative to how much that gap wobbles from prompt to prompt. We label the magnitude on the conventional scale (negligible, small, medium, large at 0.2, 0.5, 0.8). Because Cohen's d isolates effect size from sample size, it answers "how much" independently of whether the difference reached significance.

### If no effect, are models the same?

The TOST equivalence test is most useful when we find no statistically significant difference in the other tests. A finding of no difference in the mixed-effects model tells us that the difference could be due to chance or insufficient data, and it doesn't necessarily prove that the models are statistically equivalent (meaning, it doesn't matter which one we choose for a task). To make an affirmative claim of equivalence we use TOST (two one-sided tests), which inverts the usual logic: the null hypothesis becomes "the models differ by more than a margin we care about," and rejecting it lets us conclude that any difference is small enough to be practically irrelevant. Each model is tested against the reference within a margin the user sets (in standardized units, since the composite is standardized). 

## Other Tests

Two further tests are also included. A bootstrap confidence interval on the mean paired difference (resampling the prompts many times and recomputing) provides a distribution-free cross-check on the parametric intervals, making no assumptions about the shape of the data. 
We also report the skewness of each model's score distribution, because rubric scores often pile up near the top of the scale. If the LLMs being tested all perform rather well, we might see scores highly skewed, and knowing the degree of that skew helps us know what effect it has on our results. This is also one reason why the Wilcoxon signed-rank test is valuable, as it is non-parametric.



Not all tests are run with every analysis flow, for convenience. Use `--verbose` to ensure every test is produced and saved in the report. Also, if one or more LLMs fail to produce a successful response to a prompt case, that prompt case will be excluded from the Friedman/Wilcoxon tests, as these are rank based and require a response from every model in testing to be calculated.

# Usage / Getting Started {#sec-usage}

The recommended method for using this library is creating a new repository and adding this library as a dependency. The `init` command in the CLI sets up a basic example with the folder structures that the rest of the tooling expects. This structure will contain boilerplate that will run, and you can replace it with your custom data, prompts, and evals to customize and get your own project working. 


## Install

Python 3.12+ is required to begin, and `uv` is recommended as the environment management utility. There are several optional sections that you can include by adding them in square braces in the installation, as shown below with "analysis". Other choices currently include "openai" and "anthropic".

```uv add "llm-workbench[analysis]"        # or: pip install "llm-workbench[analysis]"```


## Init/Scaffold

```uv run llm-workbench init```

This copies three things into your current directory - files you now own and edit:

```
prompt.txt                              # the prompt template ({input} gets filled per row)
dataset.jsonl                           # a tiny 5-row example test set
evals_kit/example-helpfulness/          # one judge eval (rubric + judge prompt)
    eval.yaml
    prompt.md
```

## Test Run

This test will run out of the box, without model credentials, using mock. Omitting `--judge-model` means there won't be any results scores, but you'll be able to see where they are created. If you do not supply `--out-dir` it will create a folder based on the timestamp.

```
uv run llm-workbench run compare \
    --prompt-template prompt.txt --dataset dataset.jsonl \
    --model mock/echo --eval example-helpfulness --out-dir runs/smoke
```

## Connect Your LLM Backend

Put credentials in a .env in the project root (loaded automatically; real env vars win over it). Using openai/, anthropic/, or bedrock-mantle/ requires the matching install extra (see Install), but Ollama and plain Bedrock need none. Pick whichever matches what you installed:

```
# Ollama (local, free, no key)      → nothing to set; just have Ollama running
# OpenAI-compatible                 → OPENAI_API_KEY=...   (OPENAI_BASE_URL for others)
# Anthropic                         → ANTHROPIC_API_KEY=...
# Bedrock / Mantle                  → AWS_PROFILE=...  AWS_REGION=...  (SigV4, no key)
```

Confirm it actually works before a full run:

```
ollama pull llama3.1 && ollama pull mistral  # Ollama requires pulling the models first
uv run llm-workbench models ping ollama/llama3.1     # one short call, prints latency + result
```

## Run With Model Backend

If that succeeds, you're ready to run a test case with live models.

This includes two live Ollama models and a judge model, also from Ollama - use your own backend. 

```
uv run llm-workbench run compare \
    --prompt-template prompt.txt --dataset dataset.jsonl \
    --model ollama/llama3.1 --model ollama/mistral \
    --eval example-helpfulness \
    --judge-model ollama/llama3.1 \
    --out-dir runs/example
```

Every model answers every prompt, and the judge scores each answer against the rubric.

## Test Analysis

```uv run llm-workbench analyze runs/example --reference ollama/llama3.1```

This produces plots, the analysis report, and prints analyses to the command line. Add `--verbose` to get a more comprehensive set of analyses at the command line as well. The `--reference` argument indicates which model you'd like to be the comparison item that other models are examined against, but you can omit this and accept the default. 

## Eval Design Wizard {#sec-wizard}

```uv run llm-workbench evals new```

Run with no arguments, this command launches a guided wizard that builds an eval folder for you, so you don't have to write `eval.yaml` and `prompt.md` by hand. It first asks what kind of eval you want — an LLM judge, deterministic checks, or both — and that choice determines the rest of the questions. For a judge, it walks through the scoring scale (a 1–5 graded rubric, binary pass/fail, or custom levels), the judge variables the prompt fills per case (input and output by default), and the judge instructions. 

For deterministic checks, it collects one or more scorers, such as `exact_match`, `contains`, `regex`, `json_schema`, or `numeric_tolerance`, which compare the output to your dataset with no model. You may need to manually edit the `eval.yaml` file depending on which kind of scorer you picked. Scorer-specific options like a JSON schema, a regex pattern, or a numeric tolerance are added afterward by editing `eval.yaml` under the scorer's params (see @sec-appendix1 for the exact shapes), then confirmed with `evals validate`.

For scripted or CI use, the same command runs non-interactively by supplying the name and scale up front, as in:

`uv run llm-workbench evals new answer-quality --labels 1-5 --yes`

or

`uv run llm-workbench evals new exact-category --scorer exact_match:expected --yes`

Two companion commands complete the set:
* `evals validate` confirms an eval's files are well-formed (handy after you hand-edit a prompt)
* `evals list` shows the evals currently available in your project


## Calibrating Judges {#sec-calibration}

```uv run llm-workbench calibrate runs/example```

This offers a CLI interface where the user is prompted to review judge decisions one at a time. You'll see the input, the model's answer, and the judge's score and reasoning, and you enter your own score on the same rubric by typing a number. 

There are keyboard shortcuts to assist: `a` to agree with the judge, `s` to skip, `o` to see the full answer, or `q` to quit. This workflow is also resumable (your scores are saved to `review.jsonl`, already-seen decisions are skipped). 

A report is generated to show the overall results, indicating the difference between the user and the LLM's assessments of responses. High bias indicates the LLM judge is not properly aligned, likely indicating that the eval prompts need clarification, or that the judge LLM is not high enough quality to be able to do the work. 

The `--stats` flag can be used with the command to just get reporting without asking for any unreviewed cases. In addition, `--eval/--model/--scorer` flags can be used to filter the cases that are being shown for review. 

## Power Analysis
 
If you have an effect size in mind - the smallest difference you would care about detecting - you can ask directly how many prompts it takes:

```uv run llm-workbench power --effect-size 0.4```

More often, you don't know the effect size in advance. In that case, run a small pilot with the real models and evals, then let the tool read the observed effects back out of the results.

```uv run llm-workbench power --from-run runs/example --reference ollama/llama3.1```

# Case Study / Worked Example {#sec-casestudy}

The following example is a use case where an LLM is asked to extract structured data from a free text passage. I tested three LLMs on the same task, using custom evals.


After following the initialization and setup instructions shown above, I had the scaffold file structure in place. I was using Ollama for my models, so I ran Ollama locally and installed a few models.


## Check Model Availability
I checked to ensure that the models I wanted were available, and got successful results - see one example below.

```
uv run llm-workbench models ping ollama/llama3.1
```

```
✓ ollama/llama3.1  (11.09s, 21 tok)
Hello to you.
```


## Prompt Cases

At this point, I was ready to write my own case prompts and evaluations. Because my task was about extracting formatted data from a text blob, I created evals that use a comparison to a provided answer, and also LLM as a judge.

The default in `prompt.txt` provided is very bare, just 
```
Answer the question concisely.

Question: {input}
Answer:
```

I customized this and added some additional elements. 

```
Given a passage of text, extract the date, time, description, and location of the event being described.
Your response should return these values in the following format:

DATE:[date]
TIME:[time]
LOCATION:[location]
DESCRIPTION:[description]

Make the description brief, no more than 50 words. Give the date in MM-DD-YYYY format, and the time in HH:MM AM/PM.

Passage: {passage}
Answer:
```

The prompt cases `dataset.jsonl` will contain the following fields, to allow checking results according to accuracy.

```
{"id": "0005", "passage": "Circle February the 14th on your calendar (2027). Reservations are for seven in the evening at Bellamy's, the little bistro on Chestnut Street downtown - the one with the green awning. It's an anniversary dinner, so dress up a bit.", "expected_date": "02-14-2027",  "expected_time": "07:00 PM",  "expected_location": "Bellamy's on Chestnut Street"}
```

## Evaluations
My next job was to design new evals that make sense with these tasks. You can use the wizard (@sec-wizard), but for full visibility here I wrote the files myself.

First, I wanted to simply compare the date to make sure the model got this right. Second, I wanted to make sure the description was broadly correct, and I used an LLM-as-a-judge for that. To set these up, I created new folders in `evals_kit`: `date_correct` and `description_accuracy`. (I deleted the scaffold example for "helpfulness" after using it for a template.)

Each eval folder takes one `eval.yaml` and one (optional) `prompt.md`. Eval contains specifics and prompt is, naturally, the prompt the LLM judge receives.

For coverage, I made Date Correct a "contains" eval and Description a "judge" eval so the reader could see multiple possibilities.

### Deterministic
For Date Correct, I made it a very simple check- did the output from the LLM contain the value I've listed as the expected date? I was being exact, so if the LLM did not follow my formatting requirements, and gave the date in the wrong format, it would fail. This is something I had to decide on as the user, but fuzzy matching is also possible. For example, I could ask the LLM judge to compare the desired output with what the task LLM rendered.

``` 
# eval.yaml

name: Date Correctness
description: >-
  Judges whether the response contains the expected date output.

scorers:
  - type: contains
    expected_field: expected_date
```
While I didn't use the other versions of deterministic judges in this case, I included examples of how to structure them in @sec-appendix1.


### LLM Judge
For judge, I used an LLM-as-a-judge, potentially including extra facts from the dataset, but not necessarily. For this case, I had an LLM assess whether the event description matches roughly what the prompt indicated. I didn't have a ground truth in place, just the original passage. Basically, I had a second LLM do an additional pass over the question, getting a review of the first LLM's effort.

This called for both the `eval.yaml` and `prompt.md`.

```
# eval.yaml

name: Description Accuracy
description: >-
  Judges whether the extracted event description matches what the passage actually describes.
direction: MAXIMIZE
labels:
  "Accurately captures what the event is, with no errors.": 5
  "Captures the event with minor omissions or vagueness.": 4
  "Roughly right but generic or missing a notable aspect.": 3
  "Partially wrong, or conflates the event with irrelevant detail.": 2
  "Wrong event, or no usable description.": 1
prompt_file: prompt.md
```

```
# prompt.md

You are checking how well a model's extracted event DESCRIPTION matches the event actually described in a passage.

The passage below describes a single event. The model was asked to extract the date, time, location, and a brief description of that event. Judge ONLY the description: does it accurately convey what the event is (its occasion or purpose), allowing for reasonable paraphrase? Ignore the date, time, and location fields - those are scored separately. A description can be correct even if worded differently from the passage, as long as it captures the right event.

<passage>
{passage}
</passage>

<model_output>
{output}
</model_output>
```

**Custom Evals**

To use custom pattern lookups requires a little more effort, in the form of a custom task. You can use this if your LLM output contains complex formatting, where you want to look for a particular value in context, or you can use it to look for several values at once. Building this out means writing a couple of extra files. This is described in @sec-custom-evals.

## Run

Now I was ready to run the experiment. 

```
uv run llm-workbench run compare \
--prompt-template prompt.txt --dataset dataset.jsonl \
--model ollama/mistral-small3.2 --model ollama/laguna-xs-2.1:latest --model ollama/tulu3:latest \
--judge-model ollama/llama3.1 \
--out-dir runs/example_2
```

```
Comparing 3 model(s) on 2 eval(s) → runs/example_2/
models: ollama/mistral-small3.2, ollama/laguna-xs-2.1:latest, ollama/tulu3:latest   judge: ollama/llama3.1
· ollama/mistral-small3.2  0001
· ollama/mistral-small3.2  0002
· ollama/mistral-small3.2  0003
· ollama/mistral-small3.2  0004
· ollama/mistral-small3.2  0005
· ollama/laguna-xs-2.1:latest  0001
· ollama/laguna-xs-2.1:latest  0002
· ollama/laguna-xs-2.1:latest  0003
· ollama/laguna-xs-2.1:latest  0004
· ollama/laguna-xs-2.1:latest  0005
· ollama/tulu3:latest  0001
· ollama/tulu3:latest  0002
· ollama/tulu3:latest  0003
· ollama/tulu3:latest  0004
· ollama/tulu3:latest  0005
```



**Comparison summary**

| model | eval | scorer | n | mean score | pass rate | errors |
|---|---|---|---|---|---|---|
| ollama/laguna-xs-2.1:latest | Date Correctness | contains | 5 | 0.8 | 80% | 0 |
| ollama/laguna-xs-2.1:latest | Description Accuracy | judge | 5 | 4.6 | | 0 |
| ollama/mistral-small3.2 | Date Correctness | contains | 5 | 0.8 | 80% | 0 |
| ollama/mistral-small3.2 | Description Accuracy | judge | 5 | 5 | | 0 |
| ollama/tulu3:latest | Date Correctness | contains | 5 | 0.6 | 60% | 0 |
| ollama/tulu3:latest | Description Accuracy | judge | 5 | 5 | | 0 |

**Per-model reliability**

| model | runs | task errors | mean latency (s) |
|---|---|---|---|
| ollama/laguna-xs-2.1:latest | 5 | 0 | 21.86 |
| ollama/mistral-small3.2 | 5 | 0 | 14.02 |
| ollama/tulu3:latest | 5 | 0 | 4.76 |


```
✓ Done. Results in runs/example_2/results.jsonl and results.csv
```

## Calibrate Judge

```
uv run llm-workbench calibrate runs/example_2
```

Because calibrate defaults to `--scorer` judge, it only pulled up the Description Accuracy decisions; the deterministic Date check was skipped automatically. I conducted the reviews, scoring each description on the same 1–5 rubric. After all the responses were reviewed, then a summary of the result was shown in the terminal. If this showed that I disagreed significantly with the LLM judge, then that would call for either eval prompt adjustment or trying a new judge entirely.


```
✓ Reviewed 15 decision(s) this session. Saved to runs/example_2/review.jsonl
```


**Judge ↔ human calibration**

| eval | n | mean judge | mean human | bias (J−H) | exact agree | within 1 |
|---|---|---|---|---|---|---|
| Description Accuracy | 15 | 4.867 | 4.6 | +0.267 | 80% | 93% |
| ALL | 15 | 4.867 | 4.6 | +0.267 | 80% | 93% |



```
bias = mean judge − mean human (positive = judge scores higher than you).
```



## Test Analysis

Now we are ready to actually run the full analysis suite. (Note: This is presented so users can see what it will all look like, but the actual results are not particularly meaningful because only a tiny pilot size sample has been used.)

```
uv run llm-workbench analyze runs/example_2 --verbose
```


*Reliability — task completion*      

| model                | runs | errors | success rate |
|----------------------|------|--------|--------------|
| laguna-xs-2.1:latest | 5    | 0      | 100%         |
| mistral-small3.2     | 5    | 0      | 100%         |
| tulu3:latest         | 5    | 0      | 100%         |


**Analysis**  models=3  metrics=2  reference=laguna-xs-2.1:latest  alpha=0.05  margin=±0.2

Design: repeated measures, paired on record. The full statistical battery runs on the quality composite; individual metrics get mean tables.

Standardized composite: every metric (graded + binary accuracy) is z-scored within-metric, so all metrics are pooled on a common SD scale. Composite means, mixed-model coefficients, Cohen's d and the TOST margin are all in SD units (margin ±0.2 SD).


*Overall performance by model (quality metrics)*  

| model                       | mean  | std   | n |
|-----------------------------|-------|-------|---|
| mistral-small3.2            | 5.000 | 0.000 | 5 |
| tulu3:latest                | 5.000 | 0.000 | 5 |
| laguna-xs-2.1:latest  (ref) | 4.600 | 0.548 | 5 |


**Per-metric mean scores** (full tests run on the aggregate below)  
*Mean score by metric × model*

| metric                       | mistral-small3.2 | tulu3:latest | laguna-xs-2.1:latest* |
|------------------------------|------------------|--------------|-----------------------|
| Date Correctness / contains  | 0.80             | 0.60         | 0.80                  |
| Description Accuracy / judge | 5.00             | 5.00         | 4.60                  |

*\* = reference model*


**Mixed Effects Model Results**  
*quality_composite  —  mean score*                     


| model                       | mean   | median | n |
|-----------------------------|--------|--------|---|
| mistral-small3.2            | 0.271  | 0.50   | 5 |
| tulu3:latest                | 0.045  | 0.50   | 5 |
| laguna-xs-2.1:latest  (ref) | -0.317 | -0.63  | 5 |

Friedman: chi2(2)=2.00, p=0.3679     Kendall's W=0.200 (small)  (n=5 complete cases)

*Mixed model (score ~ C(model) + C(metric)) — diff vs laguna-xs-2.1:latest*      

| model            | coef   | 95% CI         | p      | real? |
|------------------|--------|----------------|--------|-------|
| mistral-small3.2 | +0.588 | [-0.37, +1.55] | 0.2301 | no    |
| tulu3:latest     | +0.362 | [-0.60, +1.32] | 0.46   | no    |


```{=latex}
\begingroup\scriptsize
```

```
                      Mixed Linear Model Regression Results
==================================================================================
Model:                      MixedLM          Dependent Variable:          score   
No. Observations:           30               Method:                      REML    
No. Groups:                 5                Scale:                       1.2019  
Min. group size:            6                Log-Likelihood:              -43.7448
Max. group size:            6                Converged:                   Yes     
Mean group size:            6.0                                                   
----------------------------------------------------------------------------------
                                          Coef. Std.Err.   z   P>|z| [0.025 0.975]
----------------------------------------------------------------------------------
Intercept                                 0.000    0.559 0.000 1.000 -1.096  1.096
C(model)[T.ollama/mistral-small3.2]       0.588    0.490 1.200 0.230 -0.373  1.549
C(model)[T.ollama/tulu3:latest]           0.362    0.490 0.739 0.460 -0.599  1.323
C(metric)[T.Description Accuracy / judge] 0.000    0.400 0.000 1.000 -0.785  0.785
Group Var                                 0.000    0.225                          
==================================================================================
```

```{=latex}
\endgroup
```

*Paired Cohen's d*

| pair                                     | mean diff | d     | magnitude |
|------------------------------------------|-----------|-------|-----------|
| laguna-xs-2.1:latest vs mistral-small3.2 | -0.588    | -0.73 | medium    |
| laguna-xs-2.1:latest vs tulu3:latest     | -0.362    | -0.33 | small     |
| mistral-small3.2 vs tulu3:latest         | +0.226    | +0.45 | small     |

*TOST equivalence vs laguna-xs-2.1:latest (margin ±0.2)*

| model            | mean diff | 90% CI         | TOST p | equivalent? |
|------------------|-----------|----------------|--------|-------------|
| mistral-small3.2 | +0.588    | [-0.18, +1.36] | 0.8291 | no          |
| tulu3:latest     | +0.362    | [-0.70, +1.42] | 0.6196 | no          |

*Score skewness (negative = ceiling skew)*    

| model                | skew  |
|----------------------|-------|
| mistral-small3.2     | -1.50 |
| tulu3:latest         | -0.41 |
| laguna-xs-2.1:latest | +0.31 |

*Bootstrap 95% CI of mean diff vs laguna-xs-2.1:latest*  

| model            | mean diff | bootstrap 95% CI |
|------------------|-----------|------------------|
| mistral-small3.2 | +0.588    | [+0.000, +1.177] |
| tulu3:latest     | +0.362    | [-0.452, +1.177] |

Reminders: a significant p with tiny Kendall's W is real but practically trivial; TOST concludes equivalence, non-significance alone does not.  
✓ Wrote runs/example_2/analysis.json and analysis.md  
✓ Wrote 2 plot(s): composite_effects.png, composite_violin.png

![](composite_effects.png)
![](composite_violin.png)

## Power Analysis

This analysis was just illustrative, however, because this test had only a few prompts. To determine what size of sample I needed to identify a meaningful effect size, I ran the power analysis. 

```
uv run llm-workbench power --from-run runs/example_2
```

The resulting terminal output told me that 76 prompts would be necessary to create a sufficiently large test. The experiment so far had only 5 prompts, meaning more would be required.

```
Pilot effects on the standardized composite (all metrics pooled) vs laguna-xs-2.1:latest. 'N needed' = prompts to reach 80% power at alpha=0.05.
```

**Required sample size (paired)**

| model | observed dz | mean diff | pilot N | power now | N needed |
|---|---|---|---|---|---|
| mistral-small3.2 | +0.730 | +0.588 | 5 | 24% | 17 |
| tulu3:latest | +0.326 | +0.362 | 5 | 9% | 76 |

```
Plan for the smallest effect: ~76 prompts to detect tulu3:latest vs reference (dz=+0.326).
```

## Custom Ground Truth Evaluations {#sec-custom-evals}

An LLM judge isn't always the best choice for a use case. When a field has an objectively correct answer stored in your dataset, you can score it deterministically. We already discussed the `contains` option, which requires minimal user customization, but this section reviews more detail about writing a custom parser.


**Step 1 - Confirm the gold value is in your dataset** 

Each record must carry the correct answer. Our dataset already includes it:

```
{"id": "0003", 
"passage": "The team offsite is happening on Thursday, April 16th, 
2026. We'll start at 9:30 in the morning ...", 
"expected_time": "09:30 AM"}
```

`expected_time` is the value the eval will grade against.

**Step 2 - Write a task that exposes the parsed answer**

The default prompt-template task returns only the raw model output. Ground-truth scoring needs the extracted time specifically, so we write a small custom task that runs the model and places the parsed value in aux - a dictionary the eval can read by name. This output should come as the provided `TaskOutput` class object.

```
# event_task.py
import re
from pathlib import Path

from llm_workbench.object_types import GenerationRequest
from llm_workbench.tasks.base import TaskOutput
from llm_workbench.evals.template import fill_template


class ExtractionTask:
    name = "event-extraction"

    def __init__(self):
        self.template = Path("prompt.txt").read_text()

    def run(self, record, client):
        prompt = fill_template(self.template, record)
        response = client.generate(GenerationRequest(prompt=prompt, max_tokens=256))
        text = response.text or ""

        # Pull the value from the "TIME:" line into aux so the eval can grade it.
        match = re.search(r"TIME:\s*(.+)", text)
        parsed_time = match.group(1).strip() if match else ""

        return TaskOutput(
            output=text,
            prompt=prompt,
            response=response,
            aux={"time": parsed_time},
        )

```

The key line is `aux={"time": parsed_time}`: whatever the task puts in aux becomes available to the eval by that name. If you create multiple ground truth evals, you can add the fields to this class and reuse it instead of creating more. Also, notice that the `prompt.txt` is called in here- if you change which prompt you're using, make sure to update it here.

**Step 3 - Create the eval**

This follows the previously laid out structure, where an eval folder contains the yaml and the prompt. `eval.yaml` declares the ground-truth comparison:

```
# evals_kit/time_correct/eval.yaml
name: Time Correctness
description: The extracted time matches the expected time.
direction: MAXIMIZE
labels:
  correct: 1
  incorrect: 0
prompt: "Fallback only - unused when expected_time is present."
ground_truth:
  expected_field: expected_time   # the gold value in each dataset record
  aux_field: time                 # the value the task placed in aux
```

The `ground_truth` block is what turns this into a deterministic accuracy check: for every record that has `expected_time`, the eval compares it to aux["time"] and skips the LLM judge entirely. We use an inline prompt placeholder because there is no prompt required for a ground truth test.

**Step 4 - Run the comparison**

Because the extraction logic now lives in the task, we pass `--task` instead of `--prompt-template`:

```
uv run llm-workbench run compare \
  --task event_task:ExtractionTask \
  --dataset dataset.jsonl \
  --model ollama/mistral-small3.2 --model ollama/laguna-xs-2.1:latest --model ollama/tulu3:latest \ 
  --judge-model ollama/llama3.1 \
  --out-dir runs/experiment
```

Two flags deserve a note. First, `--task` and `--prompt-template` are mutually exclusive - the custom task builds its own prompt (here by reading `prompt.txt`). Second, `--judge-model` is required even though the judge is bypassed: the ground-truth logic is part of the judge scorer, which is only activated when a judge model is supplied. It will not actually be called for records that carry the gold value, but the flag must be present.

The eval emits a binary metric - 1.0 when the parsed time matches, 0.0 otherwise. By default it is standardized and pooled into the quality composite alongside the judged metrics; use `--raw` for a graded-only composite, or `--exclude-metric` to report it as a standalone accuracy track.





## Custom Model Backends {#sec-custom-models}

As noted previously, this library contains several model provider backends that give users access to dozens or hundreds of LLM choices. However, this ecosystem is even more expansive and growing, so there is a system available for implementing custom model backends as well.

The essential process is writing a small client class, registering it under a prefix, and naming your module one time in your `.env` file. 

**Client**

The client file can follow this general example, importantly including the `register_provider` step. For convenience, I'd place it in the root directory of your project. The `config` argument in `from_config` is the library's settings loaded from your `.env` (the same source the built-in backends use); a custom provider reads its own credentials from the environment, as shown with `os.environ["ACME_API_KEY"]`.


```
# acme_provider.py
from llm_workbench.models.base import BaseClient
from llm_workbench.models.registry import register_provider
from llm_workbench.object_types import GenerationRequest, Usage
import os


class AcmeClient(BaseClient):
    def __init__(self, model_id: str, native: str, api_key: str):
        super().__init__(model_id)
        self.native = native          # the model title after the slash, e.g. "acme/foo" -> "foo"
        self.api_key = api_key

    @classmethod
    def from_config(cls, *, model_id, native, config):
        return cls(model_id=model_id, native=native, api_key=os.environ["ACME_API_KEY"])

    def _raw_generate(self, request: GenerationRequest):
        text = [call your provider's SDK/HTTP here]
        usage = Usage(input_tokens=[fill in from provider call], output_tokens=[fill in from provider output])
        return text, usage, None


register_provider("acme", AcmeClient)
```

**Naming**

Place this line in the `.env` file. It assumes `acme_provider.py` is importable from where the CLI runs (project root on sys.path, or an installed package).

```
LLM_WORKBENCH_PROVIDERS=acme_provider
```

After this, you should be able to call your custom backend wherever, the same way examples in this document call `ollama/llama3.1` etc.


# Limitations & Roadmap

This framework does have a few constraints on the user that should be acknowledged.

* Experiments using `llm-workbench` presume a paired prompt design (@sec-methodology). This means that you must test each LLM in your experiment on the same set of prompts to get the optimal results. 
* The design strongly encourages using multiple evals to create a composite measure of quality (@sec-methodology). Using only one eval is also not going to produce thorough results for assessing model performance.
* Users who need more complex tasks or evals will have to take some extra steps to reflect these in the experimental design (@sec-extensibility). This library is not a comprehensive tool for designing the tasks for LLMs, and evals need to be customized to reflect the user's project. 

Because this project is not designed for continuous production monitoring or observability, it does not offer an interactive GUI, and does not offer interactive mechanisms for detailed testing of models' responses to individual prompts at a granular level. As discussed earlier in this paper (@sec-related), several other tools exist in the LLM evaluation ecosystem to fill these purposes. 

Some features are on the roadmap for future development, if users would find them helpful. For example, adding more model backend support would extend the usefulness of the library to more kinds of open source models. In addition, enabling users to assign custom weighting to each eval in the larger statistical calculations is planned. 

While the project emphasizes the composite measurement of quality, users may also find individual metric level statistics and multi-objective optimization tests helpful, so these are also potential additions in the future. This work would be more relevant to eval construction and design, so somewhat tangential to the project's focus currently.

Also, adding more LLM-as-a-judge calibration tools may be a part of future development, because this is such a neglected space in the LLM evaluation area. Calibrating, testing, and monitoring the performance of our LLM judges should be taken seriously as we use these LLMs for important business decision-making.

`llm-workbench` is a new framework, filling an important gap, and we hope to grow it in the future according to user feedback and needs.



# Conclusion

Organizations increasingly depend on large language models for consequential work, and they face a growing field of models to choose from under real cost pressure. Unfortunately, tools most teams reach for don't answer the most important question: they report whether an output passed a check, or let a reviewer eyeball a handful of responses, without establishing whether one model is genuinely, measurably better than another or merely appears so by chance. `llm-workbench` exists to close that gap, allowing users to make model comparisons with a statistically grounded decision.

It runs every candidate model over the same prompts in a paired design, scores them with configurable evals and calibrated LLM judges, and applies a full statistical test suite, including significance testing, effect sizes, equivalence, and power analysis, to a composite measure of quality. It treats LLM judge calibration as a first-class step while staying deliberately lightweight and unopinionated about the task itself, so that an existing production or agentic workflow can be dropped in and evaluated. The result is a tool that will help users decide which model to trust for their task with confidence.

`llm-workbench` is most valuable when switching to a cheaper or newer model needs to be justified, or when the choice of which model to start with is uncertain. Getting started is intentionally simple and low-effort, involving an init command and a dataset of representative prompts, and adding more customization is available as projects grow in scope or complexity. As the number of available LLMs grows and the stakes of choosing among them rise, rigorous, reproducible, and calibrated evaluations grow in importance as well. This library is intended to encourage users to follow experimental best practices in this work. 



# Community & Contributing

Users are invited to make suggestions and submit issues for development consideration via Github. 

License: Apache License 2.0 — https://www.apache.org/licenses/LICENSE-2.0 



# References

Cohen, J. (1988). Statistical power analysis for the behavioral sciences ( 2nd ed.). Hillsdale, NJ: Lawrence Erlbaum.

Cronbach, L. J., & Meehl, P. E. (1955). Construct validity in psychological tests. Psychological Bulletin, 52(4), 281–302. https://doi.org/10.1037/h0040957

Efron, B. (1979). Bootstrap Methods: Another Look at the Jackknife. The Annals of Statistics, 7(1), 1–26. http://www.jstor.org/stable/2958830

Friedman, M. (1937). The Use of Ranks to Avoid the Assumption of Normality Implicit in the Analysis of Variance. Journal of the American Statistical Association, 32(200), 675–701. https://doi.org/10.1080/01621459.1937.10503522

Holm, S. (1979). A Simple Sequentially Rejective Multiple Test Procedure. Scandinavian Journal of Statistics, 6(2), 65–70. http://www.jstor.org/stable/4615733

Kendall, M. G., & Smith, B. B. (1939). The Problem of m Rankings. The Annals of Mathematical Statistics, 10(3), 275–287. http://www.jstor.org/stable/2235668

Lakens, D. (2017). Equivalence tests: A practical primer for t tests, correlations, and meta-analyses. Social Psychological and Personality Science, 8(4), 355–362. https://doi.org/10.1177/1948550617697177

Schuirmann D. J. (1987). A comparison of the two one-sided tests procedure and the power approach for assessing the equivalence of average bioavailability. Journal of Pharmacokinetics and Biopharmaceutics, 15, 657–680.

Wilcoxon, F. (1945). Individual Comparisons by Ranking Methods. Biometrics Bulletin, 1(6), 80–83. https://doi.org/10.2307/3001968

Zheng, L., Chiang, W.-L., Sheng, Y., Zhuang, S., Wu, Z., Zhuang, Y., Lin, Z., Li, Z., Li, D., Xing, E. P., Zhang, H., Gonzalez, J. E., & Stoica, I. (2023). Judging LLM-as-a-judge with MT-Bench and Chatbot Arena (arXiv:2306.05685). arXiv. https://doi.org/10.48550/arXiv.2306.05685


# Appendix 1. Deterministic Eval Examples {#sec-appendix1}

These examples are the structure for `eval.yaml` for each kind of immediately supported deterministic eval. The top level fields of the JSON "scorers" section should be `type`, `expected_field`, or `params` as designated below.

## Correctness

Accepts `type` and `expected_field`.

```
name: Date Correctness
description: Judges whether the response contains the expected date output.
scorers:
  - type: contains
    expected_field: expected_date
```

## Exact Match

Accepts `type` and `expected_field`.

```
name: Category Label
description: Judges whether the response exactly matches the expected category.
scorers:
  - type: exact_match
    expected_field: expected_category
```

## Regex

Regex does not support template populating from the input to the model, so the pattern must be independent of the prompt.

Accepts `type` and `params`. `params` contains a single string reflecting the regex to match with.

```
name: Date Format
description: Judges whether the extracted date is in MM-DD-YYYY format.
scorers:
  - type: regex
    params:
      pattern: "^\\d{2}-\\d{2}-\\d{4}$"
```

```
name: Time Format
description: Judges whether the extracted time follows HH:MM AM/PM format.
scorers:
  - type: regex
    params:
      pattern: "^(0[1-9]|1[0-2]):\\d{2} (AM|PM)$"
```

```
name: Phone Format
description: Judges whether the extracted phone number is in (XXX) XXX-XXXX format.
scorers:
  - type: regex
    params:
      pattern: "^\\(\\d{3}\\) \\d{3}-\\d{4}$"
```

```
name: Price Present
description: Judges whether the response contains a dollar amount.
scorers:
  - type: regex
    params:
      pattern: "\\$[0-9,]+\\.\\d{2}"
```

## JSON Format

Accepts `type` and `params`. `params` contains `schema`. This schema block is a standard JSON Schema, in this example using three of its keywords: `type` (the JSON kind, here object), `properties` (the fields and their types), and `required` (which of those fields must be present). All three are optional in JSON Schema. Note: the inner `type: object` is JSON Schema's own keyword.


```
name: Output Structure
description: Judges whether the response is valid JSON with the required fields.
scorers:
  - type: json_schema
    params:
      schema:
        type: object
        properties:
          date: {type: string}
          time: {type: string}
          location: {type: string}
          description: {type: string}
        required: [date, time, location, description]
```

## Numeric Tolerance
Accepts `type`, `expected_field`, and `params`. `rtol` is used for relative tolerance and `atol` for absolute tolerance, in line with NumPy syntax.

You can use `exact_match` for numerics but this allows you to accept multiple renderings (4.0, 4.000, 4) that resolve to the same numeric value. `exact_match` compares strings, so 4, 4.0, and 4.000 count as three different answers. `numeric_tolerance` instead reads the first number out of the model's output and compares it as a number to the value in `expected_field`, so equivalent renderings match.

"How close" counts as a match is set by two params, both nested under params and both defaulting to 0.0:
* `atol` — absolute tolerance: a fixed allowance in the value's own units (`atol: 0.5` accepts anything within ±0.5).
* `rtol` — relative tolerance: a fraction of the expected value (`rtol: 0.01` accepts anything within ±1%).

The output passes when |output − expected| ≤ `atol` + `rtol` × |expected| — the same rule, and the same names, as NumPy's `isclose`. Because the two add together, you can set both to mean "within 1%, but always allow at least ±0.5"; that combination also avoids the trap where a pure relative tolerance demands an exact match whenever the expected value is 0.

With both left at 0.0, the rule collapses to exact numeric equality — so the `atol: 0.0` example below behaves identically to omitting params altogether, and is shown only to make the intent explicit.

```
name: Price Accuracy
description: Judges whether the extracted price is within 1% of the expected value.
scorers:
  - type: numeric_tolerance
    expected_field: expected_price
    params:
      rtol: 0.01
```

```
name: Invoice Total
description: Judges whether the extracted invoice total matches exactly.
scorers:
  - type: numeric_tolerance
    expected_field: expected_total
    params:
      atol: 0.0
```